## jet cat for hemi

In [ ]:
henrik_jets = pl.read_parquet("/Users/bandelol/Documents/code_local/data/henrik/all_jets_one_df.parquet").cast({"time": pl.Datetime("ms")})

In [ ]:
henrik_jets

In [ ]:
jets = categorize_jets(henrik_jets, None, None, ("s", "theta"), n_init=2, force=1)

In [ ]:
from matplotlib.colors import Normalize
from cartopy import crs
fig, ax = plt.subplots(figsize=(10, 2), subplot_kw={"projection": crs.PlateCarree()})
ax.add_feature(COASTLINE)
summary = jets.group_by("time", "jet ID", maintain_order=True).agg(mean=pl.col("is_polar").mean(), std=pl.col("is_polar").std())
cmap = colormaps.bold
N = 4
norm = BoundaryNorm(np.arange(cmap.N) - 0.5, cmap.N)
for j, i in enumerate(np.random.randint(0, len(summary), size=N)):
    summ = summary[i.item()]
    jet = jets.filter(pl.col("time") == summ["time"], pl.col("jet ID") == summ["jet ID"])
    color = cmap(norm(j))
    stj = jet.filter(pl.col("is_polar") < 0.01)
    edj = jet.filter(pl.col("is_polar") >= 0.01)
    to_plot = stj[["lon", "lat"]].to_numpy().T
    ax.scatter(*to_plot, color=cmap(norm(j)), s=10, marker="x")
    to_plot = edj[["lon", "lat"]].to_numpy().T
    ax.scatter(*to_plot, color=cmap(norm(j)), s=100, marker="h")
ax.set_xlim(-180, 180)
ax.set_ylim(0, 90)
fig.colorbar(ScalarMappable(norm, cmap), ax=ax, ticks=np.arange(cmap.N))

In [ ]:
ouais = "s_low"
month = 10
cutoff = 0.5
all_jets = jets
plt.hist(extract_features(all_jets, ("is_polar", ouais), season=month).filter(pl.col("is_polar") < cutoff)[ouais], bins=40, alpha=0.5, density=True)
plt.hist(extract_features(all_jets, ("is_polar", ouais), season=month).filter(pl.col("is_polar") > cutoff)[ouais], bins=40, alpha=0.5, density=True)

In [ ]:
is_polar = is_polar_gmix(jets, ["s", "theta"], mode="week", n_init=10)

In [ ]:
xys = []
all_jets = is_polar

xys = np.array(xys)
fig, axes = plt.subplots(3, 4, figsize=(11, 9), tight_layout=True, sharex="all", sharey="all")
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list("pinkredpurple", [COLORS[2], COLORS_EXT[10], COLORS[1]])
bins = np.linspace(0, 1, 31)
for month in trange(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]
    
    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=25)
    im2 = ax.hexbin(X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=25)
    
    plt.draw()
        
    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0., 1.0
    min_v, max_v = 0.75, 1.
    scaling = np.clip(im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1)
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(4 * (1 - f(scaling)) ** 2)
    im2.set_edgecolor(colormaps.greys(scaling))
    im2 = ax.add_collection(im2)
        
    ax.set_xlabel(pair[0])
    ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis() 
        
    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(*pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T, facecolor="black", edgecolor="white", marker="X", linewidths=1, s=45)
    iax = ax.inset_axes([0.65, 0.2, 0.4, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=0.5, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=0.5, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    plt.draw()
# fig.savefig(f"{FIGURES}/jet_detection_demo/gmix_demo")